# 05 Colab NIFTY50 Batch Pipeline

Use this notebook to batch through the full `data/raw/nifty50/` CSV corpus in Colab. It downloads or caches raw intraday Parquet files, builds features for every symbol independently, then trains one cross-sectional LightGBM, one LSTM, and one GRU over the combined universe.

For a quick smoke test, set `MAX_FILES = 1` in the download or feature cells, or a small value such as `MAX_FILES = 5` in the train cell.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
import sys

BASE = Path('/content/drive/MyDrive/trading_system')
REPO_DIR = BASE / 'TradingBot26'
DATA_DIR = BASE / 'data'
MODEL_DIR = BASE / 'models'
LOG_DIR = BASE / 'logs'
REPORT_DIR = BASE / 'reports'
ARTIFACT_DIR = BASE / 'artifacts'

for path in (DATA_DIR / 'raw', DATA_DIR / 'processed', MODEL_DIR, LOG_DIR, REPORT_DIR, ARTIFACT_DIR):
    path.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / 'pyproject.toml').exists():
    raise FileNotFoundError(f'Copy or clone this repo to {REPO_DIR} before running this notebook')

%cd $REPO_DIR
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])

In [ ]:
import importlib
from dataclasses import fields, replace

import pandas as pd

repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
for module_name in ('config', 'data.loader'):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import config as config_module
from config import MarketConfig, PathConfig
from data.loader import MarketDataLoader, standardize_ohlcv

market_fields = {field.name for field in fields(MarketConfig)}
required_fields = {'local_intraday_path', 'local_intraday_pattern'}
if not required_fields.issubset(market_fields):
    raise RuntimeError(
        'Colab is importing a stale TradingBot26 config.py from '
        f'{config_module.__file__}. Pull the latest repo commit, rerun the install cell, '
        'then use Runtime > Restart runtime before rerunning this notebook.'
    )
print(f'Using TradingBot26 config from: {config_module.__file__}')

paths = PathConfig(
    root=BASE,
    raw_data_dir=DATA_DIR / 'raw',
    processed_data_dir=DATA_DIR / 'processed',
    artifact_dir=ARTIFACT_DIR,
    model_artifact_dir=MODEL_DIR,
    report_dir=REPORT_DIR,
)

## 1. Cache raw NIFTY50 files

This cell walks the `nifty50` raw folder, converts every `*_5minute.csv` file into the expected per-symbol intraday Parquet cache, and writes daily context alongside it.

In [ ]:
NIFTY50_SOURCE_DIRS = [
    DATA_DIR / 'raw' / '18',
    DATA_DIR / 'raw' / 'nifty50',
    REPO_DIR / 'data' / 'raw' / '18',
    REPO_DIR / 'data' / 'raw' / 'nifty50',
]
NIFTY50_CSV_FILES = sorted(
    {
        path
        for source_dir in NIFTY50_SOURCE_DIRS
        if source_dir.exists()
        for path in source_dir.glob('*_5minute.csv')
        if path.is_file() and not path.name.startswith('.')
    }
)

if not NIFTY50_CSV_FILES:
    print('No NIFTY50 5-minute CSV files found under the Colab raw folders.')
else:
    FORCE_REFRESH = False
    MAX_FILES = 0  # set to 1 for a quick smoke test
    processed = []
    for csv_path in NIFTY50_CSV_FILES:
        base_name = csv_path.stem.removesuffix('_5minute')
        market = MarketConfig(
            symbol=base_name,
            ticker=base_name,
            series='IDX' if base_name.upper().startswith(('NIFTY', 'BANK')) else 'EQ',
            interval='5m',
            intraday_source='local-csv',
            daily_source='intraday-resample',
            local_intraday_path=csv_path,
            lookback_days=55,
        )
        loader = MarketDataLoader(paths, market)
        intraday_result = loader.download_intraday(force_refresh=FORCE_REFRESH, source='local-csv')
        daily_result = loader.download_daily_context(force_refresh=FORCE_REFRESH)
        print(
            f"{csv_path.name} -> intraday={intraday_result.rows} rows, daily={daily_result.rows} rows, "
            f"output={intraday_result.path.name}"
        )
        processed.append((csv_path.name, intraday_result.path, daily_result.path))
        if MAX_FILES and len(processed) >= MAX_FILES:
            break
    print(f'Completed raw download for {len(processed)} NIFTY50 files.')

## 2. Build features

This cell reads the cached intraday Parquet files, loads daily context, and runs the feature pipeline for every symbol.

In [ ]:
from config import FeatureConfig, LabelConfig, MarketConfig, NormalizerConfig, PathConfig
from data.loader import MarketDataLoader
from features.pipeline import FeatureEngineeringPipeline

RAW_INTRADAY_FILES = sorted(
    {
        path
        for path in (DATA_DIR / 'raw').glob('*_5m_intraday.parquet')
        if path.is_file() and not path.name.startswith('.')
    }
)

if not RAW_INTRADAY_FILES:
    print('No 5-minute intraday Parquet files found under data/raw.')
else:
    FORCE_REFRESH = False
    MAX_FILES = 0  # set to 1 for a quick smoke test
    processed = []
    labels = LabelConfig(horizon=15, target_pct=0.005, stop_pct=0.003)
    normalizer = NormalizerConfig(window=200, min_periods=200)
    features = FeatureConfig(lag_periods=(1, 5, 10), volume_confirm_window=20, include_daily_context=True)
    for intraday_path in RAW_INTRADAY_FILES:
        base_name = intraday_path.stem.removesuffix('_5m_intraday')
        market = MarketConfig(
            symbol=base_name,
            ticker=base_name,
            series='IDX' if base_name.upper().startswith(('NIFTY', 'BANK')) else 'EQ',
            interval='5m',
            intraday_source='cached-parquet',
            daily_source='intraday-resample',
        )
        loader = MarketDataLoader(paths, market)
        processed_path = FeatureEngineeringPipeline(
            paths=paths,
            market=market,
            features=features,
            labels=labels,
            normalizer=normalizer,
        ).processed_path()
        if processed_path.exists() and not FORCE_REFRESH:
            print(f'Skipping {intraday_path.name} because {processed_path.name} already exists.')
            processed.append((intraday_path.name, processed_path))
            if MAX_FILES and len(processed) >= MAX_FILES:
                break
            continue
        daily = loader.load_daily_context() if loader.daily_path().exists() else loader.download_daily_context(force_refresh=False)
        dataset = FeatureEngineeringPipeline(
            paths=paths,
            market=market,
            features=features,
            labels=labels,
            normalizer=normalizer,
        ).run(loader.load_intraday(), daily)
        print(
            f"{intraday_path.name} -> processed={len(dataset.frame):,} rows, features={len(dataset.feature_columns):,}, "
            f"output={dataset.feature_config_path.name}"
        )
        processed.append((intraday_path.name, dataset.feature_config_path))
        if MAX_FILES and len(processed) >= MAX_FILES:
            break
    print(f'Completed feature engineering for {len(processed)} files.')

## 3. Train combined models

This cell loads all processed feature files, adds a categorical `symbol` identity column, applies one calendar-date-based 70/15/15 split across the combined universe, trains one LightGBM model with `symbol` as categorical, and trains one LSTM plus one GRU from per-symbol sequences concatenated inside each split.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch

from config import SplitConfig
from features.sequences import SequenceBuilder, SequenceDatasetArrays
from models.torch_models import GRUClassifier, LSTMClassifier
from models.torch_training import result_to_dict, train_torch_classifier, write_torch_training_metadata
from scripts.train_pooled_lightgbm import (
    combine_instruments,
    date_based_split,
    load_instruments,
    shared_feature_columns,
    train_pooled_lgbm,
)

LOOKBACK = 60
EPOCHS = 20
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
NUM_WORKERS = 2
TRAIN_LIGHTGBM = True
TRAIN_LSTM = True
TRAIN_GRU = True
MAX_FILES = 0  # set to 5 for a quick pooled smoke test
MIN_ROWS = 1000
SCOPE = 'nifty50_combined_5m'
RANDOM_SEED = 42


def build_symbol_sequences(frame, *, feature_columns, lookback):
    builder = SequenceBuilder(lookback=lookback)
    xs, ys, indexes = [], [], []
    for symbol, group in frame.groupby('symbol', sort=False, observed=True):
        group = group.sort_index()
        if len(group) < lookback:
            print(f'Skipping {symbol} sequences: only {len(group):,} rows in this split.')
            continue
        arrays = builder.build(group, feature_columns=feature_columns)
        if len(arrays.y):
            xs.append(arrays.x)
            ys.append(arrays.y)
            indexes.append(arrays.index)
    if not xs:
        raise RuntimeError('No sequence arrays could be built for this split.')
    index = pd.DatetimeIndex(np.concatenate([idx.to_numpy() for idx in indexes]))
    return SequenceDatasetArrays(
        x=np.concatenate(xs, axis=0),
        y=np.concatenate(ys, axis=0),
        index=index,
    )


def shuffle_sequences(arrays, *, seed):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(arrays.y))
    return SequenceDatasetArrays(
        x=arrays.x[order],
        y=arrays.y[order],
        index=pd.DatetimeIndex(arrays.index[order]),
    )


instruments = load_instruments(
    DATA_DIR / 'processed',
    pattern='*_5m_features.parquet',
    min_rows=MIN_ROWS,
    limit=MAX_FILES or None,
)

if len(instruments) < 2:
    print('Need at least two usable 5-minute feature Parquet files for combined training.')
else:
    split_config = SplitConfig()
    base_feature_columns = shared_feature_columns(instruments)
    if not base_feature_columns:
        raise RuntimeError('No shared numeric feature columns found across processed files.')

    combined = combine_instruments(instruments, feature_columns=base_feature_columns)
    splits = date_based_split(combined, split_config)
    symbols = [instrument.name for instrument in instruments]
    print(
        f"Combined rows={len(combined):,} symbols={len(symbols):,} shared_features={len(base_feature_columns):,} "
        f"train_end={splits.train_end_date.date()} val_end={splits.validation_end_date.date()}"
    )
    print(
        f"split rows: train={len(splits.train):,} validation={len(splits.validation):,} test={len(splits.test):,} "
        f"device={torch.device('cuda' if torch.cuda.is_available() else 'cpu')}"
    )

    lgbm_result = None
    if TRAIN_LIGHTGBM:
        lgbm_result = train_pooled_lgbm(
            instruments,
            feature_columns=base_feature_columns,
            paths=paths,
            output_name=SCOPE,
            split_config=split_config,
            include_symbol_feature=True,
        )
        print(f"lgbm_model: {lgbm_result['model_path']}")
        print('lgbm_validation:', lgbm_result['metrics']['validation'])
        print('lgbm_test:', lgbm_result['metrics']['test'])

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    effective_num_workers = NUM_WORKERS if device.type == 'cuda' else 0
    train_arrays = shuffle_sequences(
        build_symbol_sequences(splits.train, feature_columns=base_feature_columns, lookback=LOOKBACK),
        seed=RANDOM_SEED,
    )
    validation_arrays = build_symbol_sequences(
        splits.validation,
        feature_columns=base_feature_columns,
        lookback=LOOKBACK,
    )
    input_size = len(base_feature_columns)
    print(
        f"sequence rows: train={len(train_arrays.y):,} validation={len(validation_arrays.y):,} "
        f"input_size={input_size:,}"
    )

    torch_results = {}
    if TRAIN_LSTM:
        lstm = LSTMClassifier(input_size=input_size, hidden_size_1=128, hidden_size_2=64, dropout=0.3)
        lstm_result = train_torch_classifier(
            model=lstm,
            model_name=f'lstm_{SCOPE}',
            train_arrays=train_arrays,
            validation_arrays=validation_arrays,
            model_dir=MODEL_DIR,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            num_workers=effective_num_workers,
            device=device,
        )
        torch_results['lstm'] = result_to_dict(lstm_result)

    if TRAIN_GRU:
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        gru = GRUClassifier(input_size=input_size, hidden_size=128, dropout=0.3)
        gru_result = train_torch_classifier(
            model=gru,
            model_name=f'gru_{SCOPE}',
            train_arrays=train_arrays,
            validation_arrays=validation_arrays,
            model_dir=MODEL_DIR,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            num_workers=effective_num_workers,
            device=device,
        )
        torch_results['gru'] = result_to_dict(gru_result)

    combined_metadata_path = MODEL_DIR / f'training_{SCOPE}_metadata.json'
    combined_metadata = {
        'scope': SCOPE,
        'symbols': symbols,
        'interval': '5m',
        'train_end_date': str(splits.train_end_date.date()),
        'validation_end_date': str(splits.validation_end_date.date()),
        'n_train_rows': len(splits.train),
        'n_validation_rows': len(splits.validation),
        'n_test_rows': len(splits.test),
        'n_train_sequences': len(train_arrays.y),
        'n_validation_sequences': len(validation_arrays.y),
        'feature_columns': base_feature_columns,
        'lgbm_feature_columns': base_feature_columns + ['symbol'],
        'lgbm_model': str(lgbm_result['model_path']) if lgbm_result else None,
        'lgbm_metrics': str(lgbm_result['metrics_path']) if lgbm_result else None,
        'lstm_model': torch_results.get('lstm', {}).get('final_model'),
        'gru_model': torch_results.get('gru', {}).get('final_model'),
        'device': str(device),
        **torch_results,
    }
    write_torch_training_metadata(combined_metadata_path, combined_metadata)
    print(f'combined_metadata: {combined_metadata_path}')
    print('Completed combined NIFTY50 training.')


## 4. Run order

Use the notebook from top to bottom. If you only want a smoke test, set `MAX_FILES = 1` in each batch cell.

## 5. Archive outputs

Create one timestamped archive in Drive containing trained models, reports, and artifacts.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import zipfile

ARCHIVE_DIR = BASE / 'archives'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
archive_path = ARCHIVE_DIR / f'tradingbot26_outputs_{timestamp}.zip'

include_dirs = [MODEL_DIR, REPORT_DIR, ARTIFACT_DIR]
with zipfile.ZipFile(archive_path, mode='w', compression=zipfile.ZIP_DEFLATED) as archive:
    for directory in include_dirs:
        if not directory.exists():
            print(f'Skipping missing directory: {directory}')
            continue
        for path in directory.rglob('*'):
            if path.is_file():
                archive.write(path, arcname=path.relative_to(BASE))

size_mb = archive_path.stat().st_size / (1024 * 1024)
print(f'Archive written: {archive_path}')
print(f'Size: {size_mb:.2f} MB')